In [1]:
import numpy as np
import matplotlib.pyplot as plt

from weather.config import (
    Experiment,
    WeatherFixedParams,
    WeatherGridParams,
    MLPFixedParams,
    MLPGridParams,
    FitFixedParams,
    FitGridParams,
)

from weather.search import Search

from mlp.utils import (
    plot_loss,
    classification_report_binary,
    plot_roc_auc,
    plot_accuracy,
)


In [2]:
SEED = 42
np.random.seed(SEED)


In [3]:
experiment = Experiment(
    name="wind_6plus_classification",

    # =========================
    # WEATHER
    # =========================
    weather_fixed=WeatherFixedParams(
        target="wind_speed",
        target_mode="binary",
        target_threshold=6.0,
        data_dir="../data",
        skip_day=True,
        normalization="global",
        encode_wind_direction=True,
    ),

    weather_grid=WeatherGridParams(
        window_aggregation=["flatten", "aggregate"],
        window_size=3,
        input_variables=(
            "temperature",
            "humidity",
            "pressure",
            "wind_speed",
            "wind_direction",
        ),
        aggregations={
            "temperature": ("mean", "min", "max"),
            "humidity": ("mean", "min", "max"),
            "pressure": ("mean", "min", "max"),
            "wind_speed": ("mean", "max"),
            "wind_direction": ("mean",),
        },
        cities=("Vancouver",),
    ),

    # =========================
    # MLP
    # =========================
    mlp_fixed=MLPFixedParams(
        task="binary",
        beta = 0.9,
        beta2 = 0.999,
        eps = 1e-8,
        adaptive_lr=True,
        lr_decay=0.99,
    ),

    mlp_grid=MLPGridParams(
        hidden_layers=(64, 64),
        loss="binary_cross_entropy",
        activation="gelu",
        learning_rate=0.008,
        seed=SEED,
        use_bias=True,
        optimizer="momentum",
    ),

    # =========================
    # FIT
    # =========================
    fit_fixed=FitFixedParams(
        verbose=True,
        log_every=None,
        use_tqdm=True,
        one_hot_if_needed=True,
        early_stopping=True,
        patience=20,
        min_delta=0.001,
    ),

    fit_grid=FitGridParams(
        epochs=400,
        batch_size="auto",
        shuffle=False,
        val_split=0.1,
    ),
)


In [4]:
# %pdb on

search = Search()

results = search.run(experiment)

# %pdb off


> /home/matti/1. Mati/MSI - semestr 2/Sieci neuronowe/Projekt 3 - Weather prediction/weather/search.py(50)build_dataset()
     48         cfg_train = self._to_weather_config(wf, wg, split="train")
     49         pdb.set_trace()
---> 50         X_train, y_train = build_dataset(cfg_train)
     51         X_train, mu, sigma = normalize_global(X_train)
     52 

*** NameError: name 'crg_train' is not defined
WeatherConfig(data_dir='../data', split='train', target='wind_speed', target_mode='binary', target_threshold=6.0, window_size=3, skip_day=True, window_aggregation='flatten', hours_per_day=24, cities='Vancouver', input_variables='temperature', aggregations={'temperature': ('mean', 'min', 'max'), 'humidity': ('mean', 'min', 'max'), 'pressure': ('mean', 'min', 'max'), 'wind_speed': ('mean', 'max'), 'wind_direction': ('mean',)}, encode_wind_direction=True, include_city_coords=False, normalization='global', max_missing_ratio_per_day=0.3)
--KeyboardInterrupt--

KeyboardInterrupt: Interrupte


KeyboardInterrupt



In [ ]:
for run in results:
    model = run["model"]
    y_test = run["y_test"]
    y_pred = run["y_pred"]

    metrics = classification_report_binary(
        y_true=y_test,
        y_score=y_pred,
        threshold=0.5,
    )

    print("\n=== TEST METRICS ===")
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}")
    print(f"AUC      : {metrics['auc']:.4f}")

    # --- plots ---
    plot_loss(
        model.history,
        title="Wind ≥ 6 m/s | train loss",
    )

    if hasattr(model, "acc_history"):
        plot_accuracy(
            model.acc_history,
            title="Accuracy evolution",
        )

    plot_roc_auc(
        y_true=y_test,
        y_score=y_pred,
        title="ROC – Wind ≥ 6 m/s",
    )
